# IGRF-14 vs pygeodyn GOVO: Gauss coefficients

Coefficient-by-coefficient difference of the two main-field models, **GOVO − IGRF**, at a few dates.

Each coefficient is one rectangle in a triangular matrix: rows are the degree $n$, columns the order $m$,
with $h_n^m$ on the left ($-m$) and $g_n^m$ on the right ($+m$).
The colour shows the difference, either **absolute** (nT) or as a **percentage** of the IGRF coefficient, $100\,(C_\mathrm{GOVO} - C_\mathrm{IGRF}) / |C_\mathrm{IGRF}|$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from samos.shmodel import SHModel
%matplotlib inline

igrf = SHModel.read_sch_igrf()
govo = SHModel.read_pygeodyn('/project/samos/2023_gji_gnm_GOVO/2023_gji_gnm_GOVO.hdf5')

In [ ]:
# ---- settings -------------------------------------------------------------
dates    = np.array(['2000-01-01', '2010-01-01', '2015-01-01', '2020-01-01'], dtype = 'datetime64[D]')
mode     = 'absolute'          # 'absolute' [nT]  or  'percent' [% of |IGRF coefficient|]
annotate = False               # write the value inside every rectangle

In [ ]:
N = min(igrf.N_degree, govo.N_degree)

# lower-triangular (Nt, N+1, N+1) arrays, cut to the common degree
gi, hi = (c[:, :N + 1, :N + 1] for c in igrf.get_gauss_coeffs(dates))
gg, hg = (c[:, :N + 1, :N + 1] for c in govo.get_gauss_coeffs(dates))

def to_matrix(g, h):
    """(Nt, N+1, N+1) g, h -> (Nt, N, 2N+1): rows n = 1..N, columns m = -N..N (h at -m, g at +m), NaN elsewhere."""
    M = np.full((len(g), N, 2 * N + 1), np.nan)
    for n in range(1, N + 1):
        M[:, n - 1, N:N + n + 1] = g[:, n, :n + 1]                  # g_n^0 .. g_n^n
        M[:, n - 1, N - n:N]     = h[:, n, n:0:-1]                  # h_n^n .. h_n^1
    return M

C_igrf = to_matrix(gi, hi)
C_govo = to_matrix(gg, hg)
diff_abs = C_govo - C_igrf                                          # nT
with np.errstate(divide = 'ignore', invalid = 'ignore'):
    diff_pct = np.where(C_igrf != 0, 100 * diff_abs / np.abs(C_igrf), np.nan)

m_axis = np.arange(-N, N + 1)
n_axis = np.arange(1, N + 1)

def coeff_name(n, m):
    return f'g({n},{m})' if m >= 0 else f'h({n},{-m})'

# largest absolute differences at each date
for i, d in enumerate(dates):
    flat = np.nan_to_num(np.abs(diff_abs[i]), nan = -1).ravel()
    top = np.argsort(flat)[::-1][:5]
    rows, cols = np.unravel_index(top, diff_abs[i].shape)
    print(f'{d}:  ' + '   '.join(f'{coeff_name(n_axis[r], m_axis[c])} {diff_abs[i, r, c]:+7.1f} nT ({diff_pct[i, r, c]:+.1f}%)'
                                 for r, c in zip(rows, cols)))

In [ ]:
def plot_gauss_diff(mode = 'absolute', annotate = False, pct_clip = 98):
    """Correlation-style plot of GOVO - IGRF per Gauss coefficient, one panel per date."""
    if mode == 'absolute':
        D, unit = diff_abs, 'nT'
        vmax = np.nanmax(np.abs(D))
        extend = 'neither'
    elif mode == 'percent':
        D, unit = diff_pct, '%'
        # a few tiny IGRF coefficients give huge percentages: cap the colour scale at a percentile
        vmax = np.nanpercentile(np.abs(D), pct_clip)
        extend = 'both' if np.nanmax(np.abs(D)) > vmax else 'neither'
    else:
        raise ValueError("mode must be 'absolute' or 'percent'")

    ncols = 2
    nrows = int(np.ceil(len(dates) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize = (14, 4.2 * nrows), layout = 'constrained', squeeze = False)

    # cell edges, so every coefficient is its own rectangle
    m_edges = np.arange(-N - 0.5, N + 1)
    n_edges = np.arange(0.5, N + 1)

    for i, (ax, d) in enumerate(zip(axs.flat, dates)):
        mesh = ax.pcolormesh(m_edges, n_edges, np.ma.masked_invalid(D[i]), cmap = 'RdBu_r',
                             vmin = -vmax, vmax = vmax, edgecolors = 'white', linewidth = 0.6)
        ax.set_aspect('equal')
        ax.invert_yaxis()                                           # n = 1 at the top, like a matrix
        ax.set_xticks(m_axis[N % 2::2]); ax.set_yticks(n_axis)
        ax.set_xlabel(r'$m$      ($h_n^m$ left,  $g_n^m$ right)')
        ax.set_ylabel(r'degree $n$')
        ax.axvline(-0.5, color = 'k', linewidth = 0.8)              # h | g separator
        ax.set_title(str(d), fontsize = 11)
        ax.spines[:].set_visible(False)
        ax.tick_params(length = 0)

        if annotate:
            fmt = '{:.0f}' if mode == 'percent' else '{:.1f}'
            for r, n in enumerate(n_axis):
                for c, m in enumerate(m_axis):
                    v = D[i, r, c]
                    if np.isfinite(v):
                        ax.text(m, n, fmt.format(v), ha = 'center', va = 'center', fontsize = 5,
                                color = 'white' if abs(v) > 0.6 * vmax else 'black')

    for ax in axs.flat[len(dates):]:
        ax.set_visible(False)

    cbar = fig.colorbar(mesh, ax = axs, shrink = 0.6, extend = extend)
    cbar.set_label(f'GOVO − IGRF [{unit}]')
    fig.suptitle(f'Gauss coefficient differences, GOVO − IGRF ({mode})')
    plt.show()

plot_gauss_diff(mode, annotate)

### Interactive switch

Toggle between absolute and percentage differences (needs `ipywidgets`; otherwise change `mode` in the settings cell).

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(plot_gauss_diff,
                     mode = widgets.ToggleButtons(options = ['absolute', 'percent'], value = mode),
                     annotate = widgets.Checkbox(value = annotate, description = 'annotate'),
                     pct_clip = widgets.IntSlider(value = 98, min = 80, max = 100, description = 'pct clip'))
except ImportError:
    print("ipywidgets not installed: set `mode` in the settings cell and re-run the plot cell")